# Single Backbone + Ensemble ML untuk Multi-Label Classification

**Architecture**: Single Deep Learning Backbone → Embeddings via Global Average Pooling → Traditional ML Ensemble untuk Multi-Label Classification

**Pipeline:**
1. Preprocessing: YOLO person detection + crop + augmentation (sama seperti sebelumnya)
2. Single Backbone: EfficientNet-B2 untuk extract embeddings
3. Global Average Pooling: Convert feature maps ke numerical vectors
4. Traditional ML: Train ensemble models (CatBoost, XGBoost, RF, GB, LR) untuk predict JENIS dan WARNA
5. Voting Classifier: Combine predictions untuk robust final output

**Keuntungan:**
- Single backbone: Lebih efficient, shared feature extraction
- Embeddings: High-level representations dari images
- Ensemble ML: Better generalization, reduce overfitting
- Multi-label: Predict JENIS dan WARNA secara bersamaan dari embeddings yang sama

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# Traditional ML Models
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("\nTraditional ML Models:")
print("  - CatBoost")
print("  - XGBoost")
print("  - RandomForest")
print("  - GradientBoosting")
print("  - LogisticRegression")

## 2. Load Data & Class Weights

In [ ]:
train_df = pd.read_csv('train.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train data shape: {train_df.shape}")
print(f"Test data count: {len(sample_sub)}")
print("\nClass Distribution:")
print(f"JENIS:")
print(f"  Kaos:   {(train_df['jenis']==0).sum()}")
print(f"  Hoodie: {(train_df['jenis']==1).sum()}")
print(f"WARNA:")
print(f"  Merah:  {(train_df['warna']==0).sum()}")
print(f"  Kuning: {(train_df['warna']==1).sum()}")
print(f"  Biru:   {(train_df['warna']==2).sum()}")
print(f"  Hitam:  {(train_df['warna']==3).sum()}")
print(f"  Putih:  {(train_df['warna']==4).sum()}")

# Class weights untuk multi-task learning
# JENIS weights (inverse frequency dengan smoothing)
total_jenis = len(train_df)
jenis_counts = train_df['jenis'].value_counts().sort_index().values
jenis_weights = total_jenis / (2 * jenis_counts)
jenis_weights = np.power(jenis_weights, 0.75)
jenis_weights_tensor = torch.FloatTensor(jenis_weights).to(device)

# WARNA weights (class-balanced loss)
beta = 0.9999
warna_counts = train_df['warna'].value_counts().sort_index().values
effective_num = 1.0 - np.power(beta, warna_counts)
warna_weights = (1.0 - beta) / effective_num
warna_weights = warna_weights / warna_weights.sum() * len(warna_weights)
warna_weights_tensor = torch.FloatTensor(warna_weights).to(device)

print(f"\nClass Weights:")
print(f"JENIS:  Kaos={jenis_weights[0]:.4f}, Hoodie={jenis_weights[1]:.4f}")
print(f"WARNA:  Merah={warna_weights[0]:.4f}, Kuning={warna_weights[1]:.4f}, Biru={warna_weights[2]:.4f}, Hitam={warna_weights[3]:.4f}, Putih={warna_weights[4]:.4f}")

# Setup Stratified K-Fold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"\nStratified K-Fold Setup:")
print(f"  Number of folds: {n_splits}")
print(f"  Strategy: Stratified by JENIS (primary label)")
print(f"  Benefits: Prevent data leakage, robust evaluation dengan dataset kecil")

## 3. YOLO Preprocessing (Sama seperti pipeline sebelumnya)

In [ ]:
def extract_clothing_region(image_path, yolo_model, conf_threshold=0.3):
    """
    Detect person region using YOLO dan crop clothing area
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = yolo_model(img_rgb, verbose=False)
    
    best_box = None
    best_conf = 0
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            
            if cls == 0 and conf > conf_threshold and conf > best_conf:
                best_conf = conf
                best_box = box.xyxy[0].cpu().numpy()
    
    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_rgb.shape[1], x2), min(img_rgb.shape[0], y2)
        cropped = img_rgb[y1:y2, x1:x2]
        return Image.fromarray(cropped)
    else:
        return Image.fromarray(img_rgb)

yolo_model = YOLO('yolov8n.pt')
print("YOLO model loaded (conf_threshold=0.3)")

## 4. Multi-Label Dataset Class

In [ ]:
class MultiLabelDataset(Dataset):
    """
    Dataset untuk multi-label classification (JENIS + WARNA)
    Single dataset untuk both labels
    """
    def __init__(self, df, img_dir, transform=None, yolo_model=None, use_yolo=True, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.yolo_model = yolo_model
        self.use_yolo = use_yolo
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        
        img_path = None
        for ext in ['.jpg', '.png']:
            path = os.path.join(self.img_dir, f'{img_id}{ext}')
            if os.path.exists(path):
                img_path = path
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found for id {img_id}")
        
        if self.use_yolo and self.yolo_model is not None:
            image = extract_clothing_region(img_path, self.yolo_model)
        else:
            image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image, img_id
        else:
            jenis = self.df.iloc[idx]['jenis']
            warna = self.df.iloc[idx]['warna']
            return image, torch.tensor(jenis, dtype=torch.long), torch.tensor(warna, dtype=torch.long)

# Transforms - SAMA SEPERTI PIPELINE SEBELUMNYA
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Multi-label dataset class dan transforms ready")

## 5. Single Backbone Model untuk Embedding Extraction

In [ ]:
class MultiLabelEmbeddingExtractor(nn.Module):
    """
    Single Backbone (EfficientNet-B2) untuk Multi-Label Classification
    
    Process:
    1. Backbone: EfficientNet-B2 features extraction
    2. Global Average Pooling: Convert feature maps → 1408-dim embedding
    3. Multi-task heads: Separate classifiers untuk JENIS dan WARNA
    4. Return embeddings untuk traditional ML
    
    Benefits:
    - Shared feature extraction untuk both tasks
    - Embeddings capture both shape (jenis) dan color (warna) information
    - More efficient daripada separate backbones
    """
    def __init__(self, jenis_classes=2, warna_classes=5, embedding_dim=1408, pretrained=True):
        super(MultiLabelEmbeddingExtractor, self).__init__()
        
        # Single backbone untuk both tasks
        backbone = models.efficientnet_b2(pretrained=pretrained)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.embedding_dim = embedding_dim
        
        # Task-specific augmentation (training only)
        self.feature_dropout = nn.Dropout2d(0.1)
        
        # Auxiliary classifiers untuk training embeddings
        # JENIS head
        self.jenis_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, jenis_classes)
        )
        
        # WARNA head
        self.warna_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, warna_classes)
        )
    
    def forward(self, x, return_embedding=False):
        # Extract features from backbone
        features = self.features(x)
        
        # Task-specific augmentation (training only)
        if self.training:
            features = self.feature_dropout(features)
        
        # Global Average Pooling
        features = self.avgpool(features)
        embedding = torch.flatten(features, 1)
        
        if return_embedding:
            return embedding
        else:
            # Multi-task outputs
            jenis_logits = self.jenis_classifier(embedding)
            warna_logits = self.warna_classifier(embedding)
            return jenis_logits, warna_logits, embedding

backbone = MultiLabelEmbeddingExtractor(pretrained=True).to(device)
print("Single Backbone Model (EfficientNet-B2) initialized")
print(f"  Parameters: {sum(p.numel() for p in backbone.parameters()):,}")
print(f"  Embedding dimension: {backbone.embedding_dim}")
print(f"  Output: 1408-dim embeddings untuk both JENIS dan WARNA")

## 6. Stratified K-Fold Cross-Validation Setup

K-Fold akan digunakan untuk:
1. Train backbone dengan cross-validation untuk robust embeddings
2. Evaluate model performance across multiple folds
3. Prevent data leakage dengan dataset kecil (777 samples)
4. Final ensemble akan trained pada all folds' embeddings

## STAGE 1: Train Backbone dengan Stratified K-Fold Cross-Validation

Train single backbone dengan multi-task learning menggunakan K-Fold untuk:
- Robust embeddings dari multiple training perspectives
- Better generalization dengan dataset kecil
- Avoid overfitting pada single train/val split

In [ ]:
# Training setup
jenis_criterion = nn.CrossEntropyLoss(weight=jenis_weights_tensor)
warna_criterion = nn.CrossEntropyLoss(weight=warna_weights_tensor)

def train_epoch(model, loader, jenis_crit, warna_crit, optimizer, device):
    """Train multi-task model untuk 1 epoch"""
    model.train()
    total_loss = 0
    jenis_correct = 0
    warna_correct = 0
    total = 0
    
    for images, jenis_labels, warna_labels in loader:
        images = images.to(device)
        jenis_labels = jenis_labels.to(device)
        warna_labels = warna_labels.to(device)
        
        optimizer.zero_grad()
        jenis_logits, warna_logits, embeddings = model(images, return_embedding=False)
        
        # Multi-task loss
        jenis_loss = jenis_crit(jenis_logits, jenis_labels)
        warna_loss = warna_crit(warna_logits, warna_labels)
        loss = jenis_loss + warna_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, jenis_pred = torch.max(jenis_logits, 1)
        _, warna_pred = torch.max(warna_logits, 1)
        total += jenis_labels.size(0)
        jenis_correct += (jenis_pred == jenis_labels).sum().item()
        warna_correct += (warna_pred == warna_labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    jenis_acc = 100 * jenis_correct / total
    warna_acc = 100 * warna_correct / total
    return avg_loss, jenis_acc, warna_acc

def validate(model, loader, jenis_crit, warna_crit, device):
    """Validate multi-task model"""
    model.eval()
    total_loss = 0
    jenis_correct = 0
    warna_correct = 0
    exact_match = 0
    total = 0
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in loader:
            images = images.to(device)
            jenis_labels = jenis_labels.to(device)
            warna_labels = warna_labels.to(device)
            
            jenis_logits, warna_logits, embeddings = model(images, return_embedding=False)
            
            jenis_loss = jenis_crit(jenis_logits, jenis_labels)
            warna_loss = warna_crit(warna_logits, warna_labels)
            loss = jenis_loss + warna_loss
            
            total_loss += loss.item()
            
            _, jenis_pred = torch.max(jenis_logits, 1)
            _, warna_pred = torch.max(warna_logits, 1)
            total += jenis_labels.size(0)
            jenis_correct += (jenis_pred == jenis_labels).sum().item()
            warna_correct += (warna_pred == warna_labels).sum().item()
            exact_match += ((jenis_pred == jenis_labels) & (warna_pred == warna_labels)).sum().item()
    
    avg_loss = total_loss / len(loader)
    jenis_acc = 100 * jenis_correct / total
    warna_acc = 100 * warna_correct / total
    exact_match_ratio = 100 * exact_match / total
    return avg_loss, jenis_acc, warna_acc, exact_match_ratio

print("Training functions ready")

In [ ]:
def extract_embeddings(model, loader, device):
    """Extract embeddings dan labels dari multi-label data"""
    embeddings_list = []
    jenis_labels_list = []
    warna_labels_list = []
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in loader:
            images = images.to(device)
            embeddings = model(images, return_embedding=True)
            embeddings_list.append(embeddings.cpu().numpy())
            jenis_labels_list.append(jenis_labels.numpy())
            warna_labels_list.append(warna_labels.numpy())
    
    embeddings = np.vstack(embeddings_list)
    jenis_labels = np.concatenate(jenis_labels_list)
    warna_labels = np.concatenate(warna_labels_list)
    return embeddings, jenis_labels, warna_labels

print("Helper functions ready")

### 7. Train Backbone dengan Stratified K-Fold

In [ ]:
print("Training Single Backbone dengan Stratified K-Fold Cross-Validation...")
print("=" * 80)

num_epochs = 10
fold_results = []
all_embeddings = []
all_jenis_labels = []
all_warna_labels = []
all_indices = []

# K-Fold Cross-Validation
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['jenis']), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/{n_splits}")
    print(f"{'='*80}")
    
    # Create fold datasets
    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].reset_index(drop=True)
    
    print(f"Train samples: {len(fold_train_df)}, Val samples: {len(fold_val_df)}")
    print(f"Train JENIS dist: Kaos={sum(fold_train_df['jenis']==0)}, Hoodie={sum(fold_train_df['jenis']==1)}")
    print(f"Val JENIS dist: Kaos={sum(fold_val_df['jenis']==0)}, Hoodie={sum(fold_val_df['jenis']==1)}")
    
    # Create datasets and loaders
    fold_train_dataset = MultiLabelDataset(fold_train_df, 'train/train', train_transform, yolo_model, use_yolo=True)
    fold_val_dataset = MultiLabelDataset(fold_val_df, 'train/train', test_transform, yolo_model, use_yolo=True)
    
    fold_train_loader = DataLoader(fold_train_dataset, batch_size=32, shuffle=True, num_workers=0)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    # Initialize model for this fold
    fold_backbone = MultiLabelEmbeddingExtractor(pretrained=True).to(device)
    fold_optimizer = optim.AdamW(fold_backbone.parameters(), lr=0.001, weight_decay=0.01)
    fold_scheduler = optim.lr_scheduler.ReduceLROnPlateau(fold_optimizer, mode='min', factor=0.5, patience=3, verbose=False)
    
    best_fold_loss = float('inf')
    
    # Train for num_epochs
    for epoch in range(num_epochs):
        train_loss, train_jenis_acc, train_warna_acc = train_epoch(
            fold_backbone, fold_train_loader, jenis_criterion, warna_criterion, fold_optimizer, device
        )
        val_loss, val_jenis_acc, val_warna_acc, val_exact_match = validate(
            fold_backbone, fold_val_loader, jenis_criterion, warna_criterion, device
        )
        
        fold_scheduler.step(val_loss)
        
        if epoch % 2 == 0 or epoch == num_epochs - 1:
            print(f"  Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Exact Match: {val_exact_match:.2f}%")
        
        if val_loss < best_fold_loss:
            best_fold_loss = val_loss
            torch.save(fold_backbone.state_dict(), f'fold_{fold}_backbone.pth')
    
    # Load best model for this fold
    fold_backbone.load_state_dict(torch.load(f'fold_{fold}_backbone.pth'))
    fold_backbone.eval()
    
    # Extract validation embeddings for this fold
    val_embeddings, val_jenis, val_warna = extract_embeddings(fold_backbone, fold_val_loader, device)
    
    # Store results
    all_embeddings.append(val_embeddings)
    all_jenis_labels.append(val_jenis)
    all_warna_labels.append(val_warna)
    all_indices.extend(val_idx.tolist())
    
    fold_results.append({
        'fold': fold,
        'best_loss': best_fold_loss,
        'val_jenis_acc': val_jenis_acc,
        'val_warna_acc': val_warna_acc,
        'val_exact_match': val_exact_match
    })
    
    print(f"\nFold {fold} Best Results:")
    print(f"  Val Loss: {best_fold_loss:.4f}")
    print(f"  JENIS Acc: {val_jenis_acc:.2f}%")
    print(f"  WARNA Acc: {val_warna_acc:.2f}%")
    print(f"  Exact Match: {val_exact_match:.2f}%")

# Aggregate results
print(f"\n{'='*80}")
print("K-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*80}")
for result in fold_results:
    print(f"Fold {result['fold']}: Loss={result['best_loss']:.4f}, JENIS={result['val_jenis_acc']:.2f}%, WARNA={result['val_warna_acc']:.2f}%, Exact Match={result['val_exact_match']:.2f}%")

avg_jenis = np.mean([r['val_jenis_acc'] for r in fold_results])
avg_warna = np.mean([r['val_warna_acc'] for r in fold_results])
avg_exact = np.mean([r['val_exact_match'] for r in fold_results])
std_exact = np.std([r['val_exact_match'] for r in fold_results])

print(f"\nAverage Performance:")
print(f"  JENIS Accuracy: {avg_jenis:.2f}% (+/- {np.std([r['val_jenis_acc'] for r in fold_results]):.2f}%)")
print(f"  WARNA Accuracy: {avg_warna:.2f}% (+/- {np.std([r['val_warna_acc'] for r in fold_results]):.2f}%)")
print(f"  Exact Match: {avg_exact:.2f}% (+/- {std_exact:.2f}%)")
print(f"{'='*80}")

# Combine all validation embeddings (covering entire dataset)
X_all = np.vstack(all_embeddings)
y_all_jenis = np.concatenate(all_jenis_labels)
y_all_warna = np.concatenate(all_warna_labels)

# Reorder to match original train_df order
sort_idx = np.argsort(all_indices)
X_all = X_all[sort_idx]
y_all_jenis = y_all_jenis[sort_idx]
y_all_warna = y_all_warna[sort_idx]

print(f"\nCombined embeddings from all folds:")
print(f"  Shape: {X_all.shape}")
print(f"  Coverage: {len(X_all)}/{len(train_df)} samples")
print("\nThese embeddings will be used to train Traditional ML ensemble")

## STAGE 2: Prepare Data untuk Traditional ML Training

Dengan K-Fold CV, kita sudah memiliki embeddings dari semua training samples (no data leakage). Sekarang akan:
1. Train Traditional ML models pada embeddings dari K-Fold
2. Evaluate dengan same K-Fold strategy
3. Train final ensemble pada all data untuk test predictions

In [ ]:
# Generate predictions on all data untuk detailed evaluation
print("\n" + "="*80)
print("DETAILED EVALUATION: Classification Report & Confusion Matrix")
print("="*80)

# Load best backbone model (fold 1) untuk consistency
eval_backbone = MultiLabelEmbeddingExtractor(pretrained=False).to(device)
eval_backbone.load_state_dict(torch.load('fold_1_backbone.pth'))
eval_backbone.eval()

# Create full dataset loader
full_dataset = MultiLabelDataset(train_df, 'train/train', test_transform, yolo_model, use_yolo=True)
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=False, num_workers=0)

# Extract predictions
print("\nExtracting predictions from backbone model...")
all_jenis_true = []
all_warna_true = []
all_jenis_pred = []
all_warna_pred = []

with torch.no_grad():
    for images, jenis_labels, warna_labels in full_loader:
        images = images.to(device)
        jenis_logits, warna_logits, _ = eval_backbone(images, return_embedding=False)
        
        _, jenis_preds = torch.max(jenis_logits, 1)
        _, warna_preds = torch.max(warna_logits, 1)
        
        all_jenis_true.extend(jenis_labels.numpy())
        all_warna_true.extend(warna_labels.numpy())
        all_jenis_pred.extend(jenis_preds.cpu().numpy())
        all_warna_pred.extend(warna_preds.cpu().numpy())

all_jenis_true = np.array(all_jenis_true)
all_warna_true = np.array(all_warna_true)
all_jenis_pred = np.array(all_jenis_pred)
all_warna_pred = np.array(all_warna_pred)

# JENIS Classification Report
print("\n" + "="*80)
print("JENIS (Kaos vs Hoodie) - Classification Report:")
print("="*80)
jenis_labels = ['Kaos', 'Hoodie']
print(classification_report(all_jenis_true, all_jenis_pred, target_names=jenis_labels, digits=4))

# JENIS Confusion Matrix
print("\nJENIS - Confusion Matrix:")
jenis_cm = confusion_matrix(all_jenis_true, all_jenis_pred)
jenis_cm_df = pd.DataFrame(jenis_cm, index=jenis_labels, columns=jenis_labels)
print(jenis_cm_df)
print(f"\nJENIS Accuracy: {accuracy_score(all_jenis_true, all_jenis_pred)*100:.2f}%")

# WARNA Classification Report
print("\n" + "="*80)
print("WARNA - Classification Report:")
print("="*80)
warna_labels = ['Merah', 'Kuning', 'Biru', 'Hitam', 'Putih']
print(classification_report(all_warna_true, all_warna_pred, target_names=warna_labels, digits=4))

# WARNA Confusion Matrix
print("\nWARNA - Confusion Matrix:")
warna_cm = confusion_matrix(all_warna_true, all_warna_pred)
warna_cm_df = pd.DataFrame(warna_cm, index=warna_labels, columns=warna_labels)
print(warna_cm_df)
print(f"\nWARNA Accuracy: {accuracy_score(all_warna_true, all_warna_pred)*100:.2f}%")

# Exact Match Analysis
exact_match = np.sum((all_jenis_pred == all_jenis_true) & (all_warna_pred == all_warna_true))
exact_match_ratio = (exact_match / len(all_jenis_true)) * 100

print("\n" + "="*80)
print("MULTI-LABEL EXACT MATCH ANALYSIS:")
print("="*80)
print(f"Exact Match Count: {exact_match}/{len(all_jenis_true)}")
print(f"Exact Match Ratio: {exact_match_ratio:.2f}%")
print(f"\nBreakdown:")
print(f"  Both Correct:        {exact_match}")
print(f"  JENIS Wrong Only:    {np.sum((all_jenis_pred != all_jenis_true) & (all_warna_pred == all_warna_true))}")
print(f"  WARNA Wrong Only:    {np.sum((all_jenis_pred == all_jenis_true) & (all_warna_pred != all_warna_true))}")
print(f"  Both Wrong:          {np.sum((all_jenis_pred != all_jenis_true) & (all_warna_pred != all_warna_true))}")
print("="*80)

## STAGE 3: Train Traditional ML Ensemble dengan K-Fold Evaluation

Strategi K-Fold untuk Traditional ML:
1. **Evaluate dengan K-Fold**: Test each model's performance across 5 folds
2. **Train pada all data**: Final models trained pada all embeddings untuk maximum data usage
3. **Voting Classifier**: Combine 5 models per task dengan soft voting
4. **K-Fold evaluation untuk Voting**: Test voting classifier performance across folds

Keuntungan:
- CV scores show model robustness dan stability
- Final models trained pada all data maximize performance
- Standard deviation across folds indicates consistency

### 8.1 Train JENIS Ensemble Models dengan K-Fold Evaluation

In [ ]:
print("Training JENIS Traditional ML Models dengan K-Fold CV...")
print("=" * 80)

# Initialize model definitions
jenis_models_def = {
    'CatBoost': lambda: CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.03, verbose=0, random_state=42),
    'XGBoost': lambda: XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42, eval_metric='logloss'),
    'RandomForest': lambda: RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1),
    'GradientBoosting': lambda: GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42),
    'LogisticRegression': lambda: LogisticRegression(max_iter=1000, C=1.0, random_state=42)
}

jenis_cv_scores = {name: [] for name in jenis_models_def.keys()}

# K-Fold evaluation untuk each model
print("\nEvaluating models dengan K-Fold CV...")
for name, model_fn in jenis_models_def.items():
    print(f"\n{name}:")
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['jenis']), 1):
        # Get fold embeddings
        X_fold_train = X_all[train_idx]
        y_fold_train = y_all_jenis[train_idx]
        X_fold_val = X_all[val_idx]
        y_fold_val = y_all_jenis[val_idx]
        
        # Train model
        model = model_fn()
        model.fit(X_fold_train, y_fold_train)
        
        # Evaluate
        val_acc = accuracy_score(y_fold_val, model.predict(X_fold_val)) * 100
        fold_scores.append(val_acc)
        print(f"  Fold {fold}: {val_acc:.2f}%")
    
    jenis_cv_scores[name] = fold_scores
    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    print(f"  Average: {avg_score:.2f}% (+/- {std_score:.2f}%)")

# Train final models pada all data
print(f"\n{'='*80}")
print("Training final JENIS models pada ALL data...")
print(f"{'='*80}")

jenis_models = {}
for name, model_fn in jenis_models_def.items():
    print(f"\nTraining {name}...")
    model = model_fn()
    model.fit(X_all, y_all_jenis)
    
    train_acc = accuracy_score(y_all_jenis, model.predict(X_all))
    cv_avg = np.mean(jenis_cv_scores[name])
    cv_std = np.std(jenis_cv_scores[name])
    
    print(f"  Train Accuracy: {train_acc*100:.2f}%")
    print(f"  CV Accuracy: {cv_avg:.2f}% (+/- {cv_std:.2f}%)")
    
    jenis_models[name] = model
    joblib.dump(model, f'jenis_{name.lower()}.pkl')
    print(f"  Saved: jenis_{name.lower()}.pkl")

print("\nJENIS Models Training Complete!")

### 8.2 Train WARNA Ensemble Models (pada same embeddings)

In [ ]:
print("Training WARNA Traditional ML Models dengan K-Fold CV...")
print("=" * 80)

# Initialize model definitions
warna_models_def = {
    'CatBoost': lambda: CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.03, verbose=0, random_state=42),
    'XGBoost': lambda: XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42, eval_metric='mlogloss'),
    'RandomForest': lambda: RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1),
    'GradientBoosting': lambda: GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42),
    'LogisticRegression': lambda: LogisticRegression(max_iter=1000, C=1.0, random_state=42, multi_class='multinomial')
}

warna_cv_scores = {name: [] for name in warna_models_def.keys()}

# K-Fold evaluation untuk each model
print("\nEvaluating models dengan K-Fold CV...")
for name, model_fn in warna_models_def.items():
    print(f"\n{name}:")
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['jenis']), 1):
        # Get fold embeddings
        X_fold_train = X_all[train_idx]
        y_fold_train = y_all_warna[train_idx]
        X_fold_val = X_all[val_idx]
        y_fold_val = y_all_warna[val_idx]
        
        # Train model
        model = model_fn()
        model.fit(X_fold_train, y_fold_train)
        
        # Evaluate
        val_acc = accuracy_score(y_fold_val, model.predict(X_fold_val)) * 100
        fold_scores.append(val_acc)
        print(f"  Fold {fold}: {val_acc:.2f}%")
    
    warna_cv_scores[name] = fold_scores
    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    print(f"  Average: {avg_score:.2f}% (+/- {std_score:.2f}%)")

# Train final models pada all data
print(f"\n{'='*80}")
print("Training final WARNA models pada ALL data...")
print(f"{'='*80}")

warna_models = {}
for name, model_fn in warna_models_def.items():
    print(f"\nTraining {name}...")
    model = model_fn()
    model.fit(X_all, y_all_warna)
    
    train_acc = accuracy_score(y_all_warna, model.predict(X_all))
    cv_avg = np.mean(warna_cv_scores[name])
    cv_std = np.std(warna_cv_scores[name])
    
    print(f"  Train Accuracy: {train_acc*100:.2f}%")
    print(f"  CV Accuracy: {cv_avg:.2f}% (+/- {cv_std:.2f}%)")
    
    warna_models[name] = model
    joblib.dump(model, f'warna_{name.lower()}.pkl')
    print(f"  Saved: warna_{name.lower()}.pkl")

print("\nWARNA Models Training Complete!")

## STAGE 4: Create Voting Classifiers & Evaluate

Combine ensemble models menggunakan soft voting untuk robust predictions.

In [ ]:
print("Creating Voting Classifiers...")
print("=" * 70)

# Create JENIS Voting Classifier dari trained models
jenis_voting = VotingClassifier(
    estimators=[(name, model) for name, model in jenis_models.items()],
    voting='soft'
)
print("\nTraining JENIS Voting Classifier pada all data...")
jenis_voting.fit(X_all, y_all_jenis)
print(f"JENIS Voting Classifier ready")
joblib.dump(jenis_voting, 'jenis_voting.pkl')

# Create WARNA Voting Classifier dari trained models
warna_voting = VotingClassifier(
    estimators=[(name, model) for name, model in warna_models.items()],
    voting='soft'
)
print("\nTraining WARNA Voting Classifier pada all data...")
warna_voting.fit(X_all, y_all_warna)
print(f"WARNA Voting Classifier ready")
joblib.dump(warna_voting, 'warna_voting.pkl')

# Evaluate dengan K-Fold strategy
print(f"\n{'='*80}")
print("VOTING CLASSIFIER K-FOLD EVALUATION:")
print(f"{'='*80}")

jenis_voting_scores = []
warna_voting_scores = []
exact_match_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['jenis']), 1):
    X_fold_train = X_all[train_idx]
    y_fold_train_jenis = y_all_jenis[train_idx]
    y_fold_train_warna = y_all_warna[train_idx]
    X_fold_val = X_all[val_idx]
    y_fold_val_jenis = y_all_jenis[val_idx]
    y_fold_val_warna = y_all_warna[val_idx]
    
    # Create and train fold voting classifiers
    fold_jenis_voting = VotingClassifier(
        estimators=[(name, jenis_models_def[name]()) for name in jenis_models_def.keys()],
        voting='soft'
    )
    fold_warna_voting = VotingClassifier(
        estimators=[(name, warna_models_def[name]()) for name in warna_models_def.keys()],
        voting='soft'
    )
    
    fold_jenis_voting.fit(X_fold_train, y_fold_train_jenis)
    fold_warna_voting.fit(X_fold_train, y_fold_train_warna)
    
    # Predict
    jenis_pred = fold_jenis_voting.predict(X_fold_val)
    warna_pred = fold_warna_voting.predict(X_fold_val)
    
    # Calculate metrics
    jenis_acc = accuracy_score(y_fold_val_jenis, jenis_pred) * 100
    warna_acc = accuracy_score(y_fold_val_warna, warna_pred) * 100
    exact_match = np.sum((jenis_pred == y_fold_val_jenis) & (warna_pred == y_fold_val_warna))
    exact_match_ratio = (exact_match / len(y_fold_val_jenis)) * 100
    
    jenis_voting_scores.append(jenis_acc)
    warna_voting_scores.append(warna_acc)
    exact_match_scores.append(exact_match_ratio)
    
    print(f"Fold {fold}: JENIS={jenis_acc:.2f}%, WARNA={warna_acc:.2f}%, Exact Match={exact_match_ratio:.2f}%")

print(f"\n{'='*80}")
print("FINAL K-FOLD RESULTS (Voting Classifier):")
print(f"{'='*80}")
print(f"JENIS Accuracy:      {np.mean(jenis_voting_scores):.2f}% (+/- {np.std(jenis_voting_scores):.2f}%)")
print(f"WARNA Accuracy:      {np.mean(warna_voting_scores):.2f}% (+/- {np.std(warna_voting_scores):.2f}%)")
print(f"EXACT MATCH RATIO:   {np.mean(exact_match_scores):.2f}% (+/- {np.std(exact_match_scores):.2f}%)")
print(f"{'='*80}")

print("\nVoting Classifiers saved!")
print("  - jenis_voting.pkl (trained on all data)")
print("  - warna_voting.pkl (trained on all data)")

## 9. Generate Test Predictions

Extract embeddings dari test data menggunakan single backbone, lalu predict menggunakan voting classifiers.

In [ ]:
# Create test dataset
test_dataset = MultiLabelDataset(sample_sub, 'test/test', test_transform, yolo_model, use_yolo=True, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print("Extracting test embeddings...")
print("=" * 70)

def extract_test_embeddings(model, loader, device):
    """Extract embeddings dan IDs dari test data"""
    embeddings_list = []
    ids_list = []
    
    with torch.no_grad():
        for images, img_ids in loader:
            images = images.to(device)
            embeddings = model(images, return_embedding=True)
            embeddings_list.append(embeddings.cpu().numpy())
            ids_list.extend([int(img_id) for img_id in img_ids])
    
    embeddings = np.vstack(embeddings_list)
    return embeddings, ids_list

# Use best fold model untuk extract test embeddings (fold 1 by default)
# Atau bisa ensemble predictions dari all folds
print("Loading best backbone model (using fold 1)...")
best_backbone = MultiLabelEmbeddingExtractor(pretrained=False).to(device)
best_backbone.load_state_dict(torch.load('fold_1_backbone.pth'))
best_backbone.eval()

# Extract test embeddings
X_test, test_ids = extract_test_embeddings(best_backbone, test_loader, device)
print(f"Test embeddings: {X_test.shape}")
print(f"Test IDs: {len(test_ids)}")

# Predict using voting classifiers
print("\nGenerating predictions using Voting Classifiers...")
jenis_pred = jenis_voting.predict(X_test)
warna_pred = warna_voting.predict(X_test)

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'jenis': jenis_pred,
    'warna': warna_pred
})

submission.to_csv('submission_ensemble_multilabel.csv', index=False)
print("\nSubmission saved: submission_ensemble_multilabel.csv")
print(f"Total predictions: {len(submission)}")
print("\nSample predictions:")
print(submission.head(10))
print("\nPrediction distribution:")
print(f"JENIS - Kaos: {(jenis_pred==0).sum()}, Hoodie: {(jenis_pred==1).sum()}")
print(f"WARNA - Merah: {(warna_pred==0).sum()}, Kuning: {(warna_pred==1).sum()}, Biru: {(warna_pred==2).sum()}, Hitam: {(warna_pred==3).sum()}, Putih: {(warna_pred==4).sum()}")

## Pipeline Summary dengan Stratified K-Fold Cross-Validation

**Complete Single Backbone + Traditional ML Ensemble Pipeline dengan K-Fold CV:**

**Architecture Overview:**
- Single EfficientNet-B2 backbone untuk extract embeddings
- Stratified K-Fold Cross-Validation (5 folds) untuk robust training
- Same embeddings digunakan untuk both JENIS dan WARNA classification
- Traditional ML ensemble (5 models per task) untuk final predictions
- Soft voting untuk combine predictions

**STAGE 1: Train Single Backbone dengan K-Fold CV**
- Multi-task learning: Train untuk both JENIS dan WARNA simultaneously
- 5-Fold Stratified Cross-Validation pada 777 training samples
- Each fold: Train pada ~622 samples, validate pada ~155 samples
- Stratified by JENIS untuk balanced class distribution
- Shared feature extraction: Single backbone learns both shape dan color features
- Preprocessing: YOLO person detection + crop + augmentation (sama seperti sebelumnya)
- Output: 1408-dim embeddings via Global Average Pooling
- Class weights: Inverse freq untuk JENIS, Class-balanced loss untuk WARNA

**Keuntungan K-Fold dengan Dataset Kecil (777 samples):**
1. **Prevent Data Leakage**: Setiap sample menjadi validation exactly once
2. **Robust Evaluation**: Performance averaged across 5 folds, bukan single split
3. **Better Generalization**: Model trained dari multiple perspectives
4. **Maximum Data Usage**: All samples used untuk both training dan validation
5. **Reduce Overfitting**: Less sensitive to particular train/val split
6. **Confidence Intervals**: Standard deviation across folds shows model stability

**STAGE 2: Extract Embeddings dengan K-Fold**
- Extract validation embeddings dari each fold
- Combine embeddings dari all folds → cover entire dataset (777 samples)
- No data leakage: Each sample's embedding dari model yang NOT trained on it
- Same embeddings untuk both JENIS dan WARNA tasks (more efficient)

**STAGE 3: Train Traditional ML Ensemble**
- JENIS: 5 models (CatBoost, XGBoost, RF, GB, LR) pada 1408-dim embeddings
- WARNA: 5 models (CatBoost, XGBoost, RF, GB, LR) pada same 1408-dim embeddings
- Total 10 models trained pada same embedding space

**STAGE 4: Voting Classifiers**
- Soft voting: Average probabilities dari 5 models per task
- Robust predictions via ensemble consensus
- Final output: JENIS + WARNA predictions

**Key Advantages:**
- Efficient: Single backbone untuk both tasks (tidak multimodel)
- Shared learning: Embeddings capture both shape dan color information
- Traditional ML: Better interpretability, faster inference
- Ensemble: Better generalization, reduce overfitting
- Same preprocessing: YOLO + crop + augmentation consistency

**Output Files:**
- multilabel_backbone.pth (single DL backbone)
- jenis_*.pkl (5 ensemble models)
- warna_*.pkl (5 ensemble models)
- jenis_voting.pkl, warna_voting.pkl (voting classifiers)
- submission_ensemble_multilabel.csv (final predictions)

**Expected Performance:**
- Target: Exact Match Ratio > 90%
- Benefits dari ensemble: More robust daripada single model
- Benefits dari shared embeddings: More efficient daripada separate backbones